# Pre-Processing and Feature Engineering

In [229]:
import pandas as pd
import numpy as np
import joblib

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder, StandardScaler

In [230]:
DATA_PATH = Path.cwd().parent / "Dataset" / "Processed" / "cleaned_dataset.csv"
ARTIFACTS_DIR = Path.cwd().parent / "artifacts"

df = pd.read_csv(DATA_PATH)
df.head()

,Brand,Current Price,Original Price,Discount Percentage,Rating,Number OF Ratings,Model Name,Dial Shape,Strap Color,Strap Material,Touchscreen,Battery Life (Days),Bluetooth,Display Size,Weight,Weight_missing,reviews_placeholder_flag,price_logic_flag
0,noise,82990.0,89900.0,7.686318,4.0,65.0,Wrb-sw-colorfitpro4alpha-std-rgld_pnk,Missing,Missing,Missing,1,8.0,1,1.5,35 - 50 g,0,False,False
1,fire-boltt,3799.0,16999.0,77.651627,4.3,20788.0,BSW046,Missing,Missing,Silicon,1,3.5,1,1.8,50 - 75 g,0,False,False
2,boat,1999.0,7990.0,74.981227,3.8,21724.0,Wave Call,Missing,Missing,Silicon,1,8.0,1,1.7,35 - 50 g,0,False,False
3,fire-boltt,1799.0,19999.0,91.004550,4.3,13244.0,BSW053,Missing,Missing,Silicon,1,3.5,1,1.8,75g +,0,False,False
4,noise,1599.0,4999.0,68.013603,4.1,13901.0,Wrb-sw-colorfitpulsegobuzz-std-blk_blk,Missing,Missing,Other,1,8.0,1,1.7,35 - 50 g,0,False,False


In [231]:
df.shape

(347, 18)

In [232]:
df["Number OF Ratings"] = np.log1p(df["Number OF Ratings"])

In [233]:
X = df.drop(columns=["Rating", "Model Name", "price_logic_flag"])
y = df["Rating"]

In [234]:
X_train, X_test, y_train, y_test= train_test_split(X, y, random_state=42, test_size=0.2)

In [235]:
brand_freq_map = X_train["Brand"].value_counts(normalize=True).to_dict()
strap_color_freq_map = X_train["Strap Color"].value_counts(normalize=True).to_dict()

DEFAULT_BRAND_FREQ = min(brand_freq_map.values())
DEFAULT_STRAP_COLOR_FREQ = min(strap_color_freq_map.values())

X_train["Brand_freq"] = X_train["Brand"].map(brand_freq_map)
X_test["Brand_freq"] = X_test["Brand"].map(brand_freq_map).fillna(DEFAULT_BRAND_FREQ)

X_train["Strap_Color_freq"] = X_train["Strap Color"].map(strap_color_freq_map)
X_test["Strap_Color_freq"] = X_test["Strap Color"].map(strap_color_freq_map).fillna(DEFAULT_STRAP_COLOR_FREQ)

X_train = X_train.drop(columns=["Brand", "Strap Color"])
X_test = X_test.drop(columns=["Brand", "Strap Color"])

joblib.dump(brand_freq_map, ARTIFACTS_DIR / "brand_freq_map.joblib")
joblib.dump(strap_color_freq_map, ARTIFACTS_DIR / "strap_color_freq_map.joblib")
joblib.dump(DEFAULT_BRAND_FREQ, ARTIFACTS_DIR / "default_brand_freq.joblib")
joblib.dump(DEFAULT_STRAP_COLOR_FREQ, ARTIFACTS_DIR / "default_strap_color_freq.joblib")

['c:\\Users\\ASUS\\OneDrive\\Documents\\ML Project\\Smartwatches project\\artifacts\\default_strap_color_freq.joblib']

In [236]:
weight_order = ["<= 20 g", "20 - 35 g", "35 - 50 g", "50 - 75 g", "75g +"]

In [237]:
weight_encoder = OrdinalEncoder(categories=[weight_order], handle_unknown="use_encoded_value", unknown_value=-1)

X_train["Weight_ordinal"] = weight_encoder.fit_transform(X_train[["Weight"]])
X_test["Weight_ordinal"] = weight_encoder.transform(X_test[["Weight"]])

X_train = X_train.drop(columns=["Weight"])
X_test = X_test.drop(columns=["Weight"])

joblib.dump(weight_encoder, ARTIFACTS_DIR / "weight_encoder.joblib")

['c:\\Users\\ASUS\\OneDrive\\Documents\\ML Project\\Smartwatches project\\artifacts\\weight_encoder.joblib']

In [238]:
print(X_train.columns.tolist())

['Current Price', 'Original Price', 'Discount Percentage', 'Number OF Ratings', 'Dial Shape', 'Strap Material', 'Touchscreen', 'Battery Life (Days)', 'Bluetooth', 'Display Size', 'Weight_missing', 'reviews_placeholder_flag', 'Brand_freq', 'Strap_Color_freq', 'Weight_ordinal']


In [239]:
Categorical_columns = X_train.select_dtypes(include=["object"]).columns
binary_columns = X_train.select_dtypes(include="bool").columns
numerical_columns = X_train.select_dtypes(exclude=["object", "bool"]).columns

for col in ["Touchscreen", "Bluetooth"]:
    if col in numerical_columns:
        numerical_columns = numerical_columns.drop(col)
        binary_columns = binary_columns.append(pd.Index([col]))

print("Categorical:", list(Categorical_columns))
print("Numerical:", list(numerical_columns))
print("Binary:", list(binary_columns))

Categorical: ['Dial Shape', 'Strap Material']
Numerical: ['Current Price', 'Original Price', 'Discount Percentage', 'Number OF Ratings', 'Battery Life (Days)', 'Display Size', 'Weight_missing', 'Brand_freq', 'Strap_Color_freq', 'Weight_ordinal']
Binary: ['reviews_placeholder_flag', 'Touchscreen', 'Bluetooth']


# Encoding categorical values

In [240]:
encoder=OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)

X_train_encoded=encoder.fit_transform(X_train[Categorical_columns])
X_test_encoded=encoder.transform(X_test[Categorical_columns])

In [241]:
cat_col_names=encoder.get_feature_names_out(Categorical_columns)

In [242]:
X_train_encoded=pd.DataFrame(X_train_encoded, columns=cat_col_names, index=X_train.index)

X_test_encoded=pd.DataFrame(X_test_encoded, columns=cat_col_names, index=X_test.index)

X_train=pd.concat( [X_train[numerical_columns], X_train[binary_columns], X_train_encoded], axis=1)
X_test=pd.concat([X_test[numerical_columns], X_test[binary_columns], X_test_encoded], axis=1)

# Scaling Numerical Values

In [243]:
scaler=StandardScaler()

In [244]:
X_train_scaled=pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled=pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

In [245]:
X_train[numerical_columns].describe()

,Current Price,Original Price,Discount Percentage,Number OF Ratings,Battery Life (Days),Display Size,Weight_missing,Brand_freq,Strap_Color_freq,Weight_ordinal
count,277.000000,277.000000,277.000000,277.000000,277.000000,277.000000,277.000000,277.000000,277.000000,277.000000
mean,10238.956679,15472.898917,47.607689,6.776154,13.753981,1.394946,0.368231,0.085795,0.247768,1.617329
std,14882.136520,16173.417170,22.482703,2.725193,7.418536,0.518124,0.483198,0.025381,0.077011,1.199995
min,1199.000000,1899.000000,0.003040,0.693147,0.750000,0.100000,0.000000,0.018051,0.075812,0.000000
25%,1999.000000,5999.000000,31.580610,4.700480,8.000000,1.300000,0.000000,0.079422,0.241877,1.000000
50%,3599.000000,7999.000000,50.012503,6.904751,14.077392,1.500000,0.000000,0.090253,0.288809,1.000000
75%,11999.000000,19990.000000,66.677780,8.703341,22.000000,1.700000,1.000000,0.097473,0.303249,2.000000
max,89490.000000,96390.000000,88.340695,13.336072,22.000000,3.000000,1.000000,0.122744,0.303249,4.000000


## Sanitize the column

In [246]:
def sanitize_columns(cols):
    return (cols.str.replace("[", "", regex=False)
                .str.replace("]", "", regex=False)
                .str.replace("<", "less than", regex=False)
                .str.replace(">", "greater than", regex=False)
                .str.replace("=", "equal to", regex=False))

X_train.columns = sanitize_columns(X_train.columns)
X_test.columns = sanitize_columns(X_test.columns)

### Save Splits


In [247]:
SPLITS_DIR = Path.cwd().parent / "Dataset" / "split"
ARTIFACTS_DIR = Path.cwd().parent / "artifacts"

In [248]:
X_train.to_csv(SPLITS_DIR / "X_train.csv", index=False)
X_test.to_csv(SPLITS_DIR / "X_test.csv", index=False)
y_train.to_csv(SPLITS_DIR / "y_train.csv", index=False)
y_test.to_csv(SPLITS_DIR / "y_test.csv", index=False)
X_train_scaled.to_csv(SPLITS_DIR / "X_train_scaled.csv", index=False)
X_test_scaled.to_csv(SPLITS_DIR / "X_test_scaled.csv", index=False)


### Save Encoder, Scaler feature order..

In [249]:
joblib.dump(encoder, ARTIFACTS_DIR / "encoder.joblib")
joblib.dump(scaler, ARTIFACTS_DIR / "scaler.joblib")

joblib.dump(list(X_train.columns), ARTIFACTS_DIR / "feature_order.joblib")
joblib.dump(list(Categorical_columns), ARTIFACTS_DIR / "categorical_columns.joblib")
joblib.dump(list(numerical_columns), ARTIFACTS_DIR / "numerical_columns.joblib")
joblib.dump(list(binary_columns), ARTIFACTS_DIR / "binary_columns.joblib")

['c:\\Users\\ASUS\\OneDrive\\Documents\\ML Project\\Smartwatches project\\artifacts\\binary_columns.joblib']

In [250]:
joblib.dump(brand_freq_map, ARTIFACTS_DIR / "brand_freq_map.joblib")
joblib.dump(strap_color_freq_map, ARTIFACTS_DIR / "strap_color_freq_map.joblib")

joblib.dump(DEFAULT_BRAND_FREQ, ARTIFACTS_DIR / "default_brand_freq.joblib")
joblib.dump(DEFAULT_STRAP_COLOR_FREQ, ARTIFACTS_DIR / "default_strap_color_freq.joblib")

['c:\\Users\\ASUS\\OneDrive\\Documents\\ML Project\\Smartwatches project\\artifacts\\default_strap_color_freq.joblib']